# Fine-tune Qwen3-VL-2B with LoRA on CORD (approach C)

Runs top to bottom on a free **Kaggle T4** (Settings: GPU T4 x2, Internet on) or Colab T4. It:

1. clones the public repo and downloads CORD-v2 (train/validation/test) and the SROIE test split,
2. trains a LoRA adapter on the 800 CORD training receipts, checkpointing as it goes,
3. evaluates the fine-tuned model with the same code as the laptop runs (CORD validation + test,
   SROIE, the four corruptions), plus T4 latency for the zero-shot model and Donut,
4. zips predictions, training log and adapter into `results.zip` in the working directory.

**Bringing results back:** download `results.zip`, unzip it in the repo root (it contains
`results/...` and `adapters/qwen3vl_lora/`), then run `make report`.

**Resuming:** checkpoints go to `ckpt/` every 25 optimizer steps and every eval appends one line per
document, so re-running the notebook in the same session continues where it stopped. To continue
in a new Kaggle session, add the previous version's output as an input and set `RESUME_DIR` to its
`ckpt` folder.

In [ ]:
import os, sys, json, time, shutil, subprocess, zipfile
from pathlib import Path

DRY_RUN = os.environ.get("DRY_RUN") == "1"  # tiny random model, 2 steps: checks the code path on a laptop
ON_KAGGLE = Path("/kaggle/working").exists()
WORK = Path("/kaggle/working" if ON_KAGGLE else "/content" if Path("/content").exists() else os.environ.get("WORK", "."))
REPO_URL = "https://github.com/egkiastas/doc-extraction-vlm"
REPO = WORK / "doc-extraction-vlm"
CKPT, ADAPTER = WORK / "ckpt", WORK / "adapter"
RESUME_DIR = ""          # e.g. "/kaggle/input/<previous-output>/ckpt"

EPOCHS, LR, GRAD_ACCUM, LORA_R = 2, 2e-4, 8, 16
RUN_SROIE = True
RUN_ROBUSTNESS = True
RUN_ZERO_SHOT_T4 = True  # zero-shot on the T4 too: GPU latency for approach B

# Estimates for one T4, not measurements: training assumes ~3 s per receipt at 1.3 MP
# (fp16 autocast, gradient checkpointing); evals assume ~25 generated tokens/s.
STAGES = [
    ("setup + data download (~2.3 GB)", 6),
    ("LoRA fine-tuning, 800 receipts x 2 epochs", 80),
    ("fine-tuned: CORD validation + test (200)", 25),
    ("fine-tuned: SROIE (100)", 12),
    ("fine-tuned: 4 corruptions x 40", 20),
    ("zero-shot on T4: CORD test (100)", 30),
    ("Donut on T4: 20 receipts", 2),
]
print(f"{'stage':48s} est. minutes")
for name, minutes in STAGES:
    print(f"{name:48s} {minutes:>5d}")
print(f"{'total':48s} {sum(m for _, m in STAGES):>5d}   (~{sum(m for _, m in STAGES) / 60:.1f} h)")

In [ ]:
def sh(cmd):
    print("$", cmd, flush=True)
    subprocess.run(cmd, shell=True, check=True)

if not DRY_RUN:
    if not REPO.exists():
        sh(f"git clone --depth 1 {REPO_URL} {REPO}")
    # Kaggle/Colab already ship torch, pydantic, numpy, scipy, pillow, pyarrow, matplotlib.
    sh(f"{sys.executable} -m pip install -q transformers==5.24.0 peft==0.22.0 accelerate==1.16.0 zss==1.2.0 "
       "pypdfium2 pytesseract 'huggingface-hub>=1.0,<2'")
    sh(f"{sys.executable} -m pip install -q --no-deps -e {REPO}")
os.chdir(REPO)
sys.path.insert(0, str(REPO))
import torch
print("torch", torch.__version__, "| cuda", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

In [ ]:
t0 = time.time()
if not DRY_RUN:
    sh(f"{sys.executable} -m docextract.data --train")
print(f"data ready in {time.time() - t0:.0f}s")

In [ ]:
from transformers import AutoModelForImageTextToText, AutoProcessor
from docextract.config import load_config
from docextract.data import load_split
from docextract.schema import PROMPT, cord_to_raw
from docextract.vlm import shrink

cfg = load_config()
MODEL = "trl-internal-testing/tiny-Qwen3VLForConditionalGeneration" if DRY_RUN else cfg.vlm_model
processor = AutoProcessor.from_pretrained(MODEL)
tok = processor.tokenizer

examples = load_split("cord-test" if DRY_RUN else "cord-train", limit=4 if DRY_RUN else None)
train = [(e.image_bytes, cord_to_raw(e.cord)) for e in examples]
# Corrections from the review queue (data/corrections/*.json) are labelled examples too.
for f in sorted((REPO / "data" / "corrections").glob("*.json")):
    d = json.loads(f.read_text())
    train.append(((f.parent / d["image"]).read_bytes(), d["target"]))
print(len(train), "training examples")

# CORD has no merchant/date/currency labels. Training on "null" for them would teach the model
# that those fields never exist (and wreck them on other receipts), so their tokens get no loss.
UNLABELLED = '"merchant": null, "date": null, "currency": null, '

def encode(image_bytes, target):
    import io
    from PIL import Image
    image = shrink(Image.open(io.BytesIO(image_bytes)).convert("RGB"), cfg.vlm_max_pixels)
    msgs = [{"role": "user", "content": [{"type": "image", "image": image},
                                         {"type": "text", "text": PROMPT}]}]
    enc = processor.apply_chat_template(msgs, add_generation_prompt=True, tokenize=True,
                                        return_dict=True, return_tensors="pt")
    answer = json.dumps(target, ensure_ascii=False) + "<|im_end|>\n"
    ans = tok(answer, add_special_tokens=False, return_offsets_mapping=True)
    masked = (1, 1 + len(UNLABELLED)) if answer.startswith("{" + UNLABELLED) else (0, 0)
    labels = [-100 if masked[0] <= s and e <= masked[1] else t
              for t, (s, e) in zip(ans.input_ids, ans.offset_mapping)]
    n_prompt, n_ans = enc["input_ids"].shape[1], len(ans.input_ids)
    batch = dict(enc)
    batch["input_ids"] = torch.cat([enc["input_ids"], torch.tensor([ans.input_ids])], 1)
    batch["attention_mask"] = torch.ones_like(batch["input_ids"])
    if "mm_token_type_ids" in enc:
        batch["mm_token_type_ids"] = torch.cat(
            [enc["mm_token_type_ids"], torch.zeros(1, n_ans, dtype=torch.long)], 1)
    batch["labels"] = torch.tensor([[-100] * n_prompt + labels])
    return batch

class Receipts(torch.utils.data.Dataset):
    def __len__(self):
        return len(train)
    def __getitem__(self, i):
        return encode(*train[i])

b = encode(*train[0])
print({k: tuple(v.shape) for k, v in b.items()},
      "| trained answer tokens:", int((b["labels"] != -100).sum()))

In [ ]:
from peft import LoraConfig, get_peft_model
from transformers import Trainer, TrainingArguments
from transformers.trainer_utils import get_last_checkpoint

if RESUME_DIR and not CKPT.exists():
    shutil.copytree(RESUME_DIR, CKPT)

if (ADAPTER / "adapter_config.json").exists():
    print("adapter already trained, skipping")
else:
    cuda = torch.cuda.is_available()
    half = torch.bfloat16 if cuda and torch.cuda.is_bf16_supported() else torch.float16
    # fp32 master weights + fp16 autocast: the T4 has no bf16, and an fp16 residual stream
    # is where overflow tends to start. 2B params in fp32 is ~8.5 GB, fine on a 15 GB T4.
    model = AutoModelForImageTextToText.from_pretrained(MODEL, dtype=torch.float32)
    model = get_peft_model(model, LoraConfig(
        r=LORA_R, lora_alpha=2 * LORA_R, lora_dropout=0.05, task_type="CAUSAL_LM",
        # language model only; the vision tower stays frozen
        target_modules=r".*language_model.*\.(q_proj|k_proj|v_proj|o_proj|gate_proj|up_proj|down_proj)"))
    model.print_trainable_parameters()
    args = TrainingArguments(
        output_dir=str(CKPT), per_device_train_batch_size=1, gradient_accumulation_steps=GRAD_ACCUM,
        num_train_epochs=EPOCHS, max_steps=2 if DRY_RUN else -1, learning_rate=LR,
        lr_scheduler_type="cosine", warmup_steps=0.03, weight_decay=0.0,
        fp16=cuda and half == torch.float16, bf16=cuda and half == torch.bfloat16,
        gradient_checkpointing=cuda, gradient_checkpointing_kwargs={"use_reentrant": False},
        logging_steps=1 if DRY_RUN else 5, save_steps=25, save_total_limit=2,
        report_to="none", remove_unused_columns=False, dataloader_num_workers=0 if DRY_RUN else 2,
        seed=0, use_cpu=DRY_RUN)
    trainer = Trainer(model=model, args=args, train_dataset=Receipts(),
                      data_collator=lambda rows: rows[0])
    t0 = time.time()
    trainer.train(resume_from_checkpoint=get_last_checkpoint(str(CKPT)) if CKPT.exists() else None)
    print(f"training took {(time.time() - t0) / 60:.1f} min")
    model.save_pretrained(ADAPTER)
    log_dir = REPO / "results" / "finetune"
    log_dir.mkdir(parents=True, exist_ok=True)
    (log_dir / "qwen3vl_lora_log.json").write_text(json.dumps(
        {"train_seconds": time.time() - t0, "epochs": EPOCHS, "lr": LR, "lora_r": LORA_R,
         "examples": len(train), "log_history": trainer.state.log_history}, indent=1))
    del model, trainer
    torch.cuda.empty_cache() if cuda else None

In [ ]:
# Same evaluation code as the laptop runs; each call appends to results/predictions/...
os.environ["VLM_ADAPTER"] = str(ADAPTER)
if DRY_RUN:
    os.environ["VLM_MODEL"] = MODEL
    os.environ["VLM_MAX_NEW_TOKENS"] = "16"

def run_eval(*args):
    t0 = time.time()
    sh(f"{sys.executable} -m evaluation.run_eval " + " ".join(args) + (" --limit 2" if DRY_RUN else ""))
    print(f"  {(time.time() - t0) / 60:.1f} min")

for split in ["cord-validation", "cord-test"] + (["sroie-test"] if RUN_SROIE else []):
    run_eval("--approach qwen3vl_lora", f"--split {split}")
if RUN_ROBUSTNESS:
    for c in ["rotate", "blur", "lowres", "jpeg"]:
        run_eval("--approach qwen3vl_lora --split cord-test", f"--corruption {c}",
                 "" if DRY_RUN else "--limit 40")

In [ ]:
# GPU latency for the zero-shot VLM and Donut on a T4 (files end in -cuda, so they never
# overwrite the laptop runs).
if torch.cuda.is_available():
    if RUN_ZERO_SHOT_T4:
        run_eval("--approach qwen3vl --split cord-test --device cuda")
    run_eval("--approach donut --split cord-test --device cuda --limit 20")

In [ ]:
# Copy the adapter next to the results and zip both.
dest = REPO / "adapters" / "qwen3vl_lora"
if ADAPTER.exists():
    shutil.copytree(ADAPTER, dest, dirs_exist_ok=True)
out = WORK / "results.zip"
with zipfile.ZipFile(out, "w", zipfile.ZIP_DEFLATED) as z:
    for base in ["results/predictions", "results/finetune", "adapters"]:
        for f in sorted((REPO / base).rglob("*")):
            if f.is_file():
                z.write(f, f.relative_to(REPO))
print(out, f"{out.stat().st_size / 1e6:.1f} MB")